# Create Virtual Icechunk Store from ERA5 Single-Levels NetCDF files (ARCO-ERA5)

Uses VirtualiZarr to create an Icechunk store pointing at ERA5 single-level NetCDF files
from the [ARCO-ERA5](https://cloud.google.com/storage/docs/public-datasets/era5) dataset
hosted publicly on Google Cloud Storage.

**No credentials required** — the files are publicly accessible via anonymous HTTPS.

**File layout** on GCS:
```
gs://gcp-public-data-arco-era5/raw/date-variable-single_level/{YYYY}/{MM}/{DD}/{variable}/surface.nc
```
Each file contains **24 hourly timesteps** for **one day** and **one variable** on a global
0.25° grid (721 × 1440).  Coverage: **1940 – 2024**, ~50 MB per file.

**Workflow**:
1. Build HTTPS file URLs for the chosen variables and date range.
2. Virtualize each NetCDF3 file with VirtualiZarr and concatenate along `time`.
3. Write an Icechunk repository (on protocoast S3) with virtual chunks pointing at the GCS HTTPS URLs.
4. Verify by opening the store with xarray and plotting a 2-m temperature map.

In [ ]:
import warnings
import os

import fsspec
import s3fs
import pandas as pd
import xarray as xr
from dotenv import load_dotenv

import icechunk
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import NetCDF3Parser
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import HTTPStore

warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# ── Variables to include ───────────────────────────────────────────────────────
# Full list of 100 variables: gs://gcp-public-data-arco-era5/raw/date-variable-single_level/
VARIABLES = [
    '2m_temperature',
    'mean_sea_level_pressure',
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
]

# ── Date range (daily files, 24 hourly timesteps each) ────────────────────────
DATE_START = '2020-01-01'
DATE_END   = '2020-01-07'   # one week; extend to e.g. '2020-12-31' for a full year

dates = pd.date_range(DATE_START, DATE_END, freq='D')
print(f'Dates: {len(dates)} days  ({DATE_START} → {DATE_END})')
print(f'Files: {len(dates) * len(VARIABLES)} total  ({len(VARIABLES)} variables × {len(dates)} days)')

# ── ARCO-ERA5 source (public GCS, anonymous HTTPS) ────────────────────────────
GCS_HTTP_BASE = 'https://storage.googleapis.com/gcp-public-data-arco-era5'
GCS_HTTP_ROOT = 'https://storage.googleapis.com/'   # VirtualChunkContainer prefix

def gcs_url(date, variable):
    return (
        f'{GCS_HTTP_BASE}/raw/date-variable-single_level/'
        f'{date.year}/{date.month:02d}/{date.day:02d}/{variable}/surface.nc'
    )

# ── Icechunk store (protocoast S3) ────────────────────────────────────────────
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)
storage_endpoint = os.environ['ENDPOINT_URL']
storage_bucket   = 'protocoast-data'
icechunk_name    = 'era5-sl-icechunk-v1'

## Step 1 — Inspect one source file

Confirm variable names, dimensions, and scale/offset encoding before virtualizing.

In [ ]:
sample_url = gcs_url(dates[0], VARIABLES[0])
print('Sample URL:', sample_url)
with fsspec.open(sample_url, 'rb') as f:
    ds_sample = xr.open_dataset(f)
print(ds_sample)
ds_sample

## Step 2 — Set up VirtualiZarr registry and Icechunk config

- **VirtualiZarr** uses an `HTTPStore` (no auth) to read metadata from the public GCS HTTPS endpoint.
- **Icechunk** stores its metadata on protocoast S3; the virtual chunks reference the same HTTPS URLs via an `http_store` virtual chunk container.
- No CDS account or Google credentials are needed.

In [ ]:
# VirtualiZarr: HTTPStore for reading NetCDF3 metadata from public GCS
http_store = HTTPStore.from_url('https://storage.googleapis.com')
registry   = ObjectStoreRegistry({GCS_HTTP_ROOT: http_store})
parser     = NetCDF3Parser()

# Icechunk store on protocoast S3
storage = icechunk.s3_storage(
    bucket=storage_bucket,
    prefix=f'icechunk/{icechunk_name}',
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True,
)

config = icechunk.RepositoryConfig.default()

# Virtual chunks are served over anonymous HTTPS — no signing required
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=GCS_HTTP_ROOT,
        store=icechunk.http_store(),
    )
)

credentials = icechunk.containers_credentials({GCS_HTTP_ROOT: None})

print('Config ready.')
print(f'  Store  : s3://{storage_bucket}/icechunk/{icechunk_name}')
print(f'  Source : {GCS_HTTP_BASE}')

## Step 3 — Virtualize and write to Icechunk

Each file is opened as a virtual dataset (metadata only, no data downloaded).
Files are merged per-day across variables, then concatenated along `time` across all days,
before writing in a single Icechunk commit.

In [ ]:
print('Virtualizing files...')
daily_vds = []

for date in dates:
    var_vds = [
        open_virtual_dataset(
            gcs_url(date, var),
            parser=parser,
            registry=registry,
            loadable_variables=['time', 'latitude', 'longitude'],
        )
        for var in VARIABLES
    ]
    # Merge all variables for this day into one dataset
    daily_vds.append(xr.merge(var_vds, compat='override', combine_attrs='override'))
    print(f'  {date.date()} done')

# Concatenate across days
ds_virtual = xr.concat(daily_vds, dim='time', coords='minimal', compat='override', combine_attrs='override')
print()
print(ds_virtual)

In [ ]:
# Delete any pre-existing repo at this path, then create fresh
icechunk_prefix = f'icechunk/{icechunk_name}'
fs_proto = s3fs.S3FileSystem(endpoint_url=storage_endpoint)
if fs_proto.exists(f'{storage_bucket}/{icechunk_prefix}'):
    fs_proto.rm(f'{storage_bucket}/{icechunk_prefix}', recursive=True)
    print(f'Deleted existing repo at s3://{storage_bucket}/{icechunk_prefix}')

repo    = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)
session = repo.writable_session('main')

t0 = pd.Timestamp(ds_virtual.time.values[0]).date()
t1 = pd.Timestamp(ds_virtual.time.values[-1]).date()

print(f'Writing {len(ds_virtual.time)} hourly time steps to Icechunk...')
ds_virtual.virtualize.to_icechunk(session.store)

msg = f'Initialized: ERA5 single-levels {t0} to {t1} ({len(VARIABLES)} vars)'
snapshot_id = session.commit(msg)
print(f'Committed: "{msg}"  (snapshot {snapshot_id})')

## Step 4 — Verify — open from Icechunk and read actual data

In [ ]:
history = repo.ancestry(branch='main')
latest  = next(history)
print(f'Latest commit [{latest.written_at}]: {latest.message}')

session_ro = repo.readonly_session('main')
ds_check   = xr.open_zarr(session_ro.store, consolidated=False, chunks={})
print(ds_check)
ds_check

In [ ]:
import hvplot.xarray

# Load one 2-m temperature snapshot to confirm virtual chunk access works
t2m = ds_check['t2m'].isel(time=0).load() - 273.15   # K → °C
t2m.attrs['units'] = '°C'

t2m.hvplot(
    x='longitude', y='latitude',
    rasterize=True,
    geo=True,
    tiles='OSM',
    cmap='RdBu_r',
    clabel='2-m Temperature (°C)',
    frame_width=800,
    frame_height=450,
    title=f'ERA5 2-m Temperature — {pd.Timestamp(ds_check.time.values[0]).strftime("%Y-%m-%d %H:%M UTC")}',
)